# **Constraint Programming and Scheduling**

Constraint satisfaction problems appear whenever we must assign choices while respecting rules. Typical applications include:

- **University timetabling:** assign courses to rooms and time slots without collisions.
- **Resource allocation:** assign machines, rooms, or equipment to tasks that compete for the same resource.
- **Frequency assignment:** assign radio frequencies to transmitters so that nearby transmitters do not interfere.
- **Map coloring:** assign colors to neighboring regions so that adjacent regions are distinguishable.

The map-coloring example is small and visual, but it has the same structure as these real problems. This notebook first develops the mathematical model and the backtracking algorithm, then translates them into Python.

## **1. The mathematical problem**

We want to assign a color to each of four regions while respecting rules about which assignments are allowed.

Let the set of regions be:

$$\mathcal{X} = \{A, B, C, D\}.$$

Here, each element $X_i \in \mathcal{X}$ is a **variable**. In this example, the variables are the four regions: $X_1=A$, $X_2=B$, $X_3=C$, and $X_4=D$.

Each variable has a **domain**, which is the set of values that the variable is allowed to take. We write the domain of variable $X_i$ as $D_i$:

$$D_i = \{red, blue, green\}.$$

Therefore, $D_A$, $D_B$, $D_C$, and $D_D$ are all the set $\{red, blue, green\}$. For example, $D_A$ is the set of colors that region A is allowed to receive.

A complete assignment selects one value from each domain. Let $a$ denote that assignment, so $a(X_i) \in D_i$ is the color selected for region $X_i$.

A **constraint** is a rule that restricts which domain values may be selected together. If regions $X_i$ and $X_j$ share a border, the constraint is:

$$a(X_i) \ne a(X_j).$$

Both $a(X_i)$ and $a(X_j)$ are colors from the domains already defined above. The complete set of these constraints depends on the borders in the map. We will identify those borders in the next section. The goal is to find a complete assignment $a$ that satisfies every constraint.

## **2. From the map to the constraints**

The variables and domains are now defined. The remaining question is: **which pairs of regions are constrained to have different colors?**

The answer comes from the geometry of the map. Two regions are related by a constraint when they share a border. The image below shows the map whose borders we will translate into mathematical constraints.

<img src="csp_four_region_map_uncolored.svg" alt="Unsolved Map" width="400">

### **2.1 Identify the neighboring pairs**

A neighbor is a region that shares a border with another region. For this map, the neighboring pairs are **unordered pairs**:

$\{A,B\}$, $\{A,C\}$, $\{B,C\}$, $\{B,D\}$, and $\{C,D\}$.

Each pair produces one constraint:

$A \ne B$, $A \ne C$, $B \ne C$, $B \ne D$, and $C \ne D$.

The neighbor relation is symmetric: if A is a neighbor of B, then B is also a neighbor of A. We list each unordered pair only once because $A \ne B$ and $B \ne A$ express the same constraint. In the Python dictionary, however, both directions are stored so that the algorithm can check neighbors from either region.

## **3. Formal model and backtracking algorithm**

A constraint satisfaction problem is described by:

$$CSP = \langle \mathcal{X}, \mathcal{D}, \mathcal{C} \rangle$$

- $\mathcal{X}$ is the set of variables.
- $\mathcal{D}$ is the collection of domains.
- $\mathcal{C}$ is the set of constraints.

For this map, a solution is a complete assignment such as:

$$A=red,\quad B=blue,\quad C=green,\quad D=red.$$

It is valid because every neighboring pair has different colors, while A and D may share red because they are not neighbors.

### **3.1 Backtracking pseudocode**

The algorithm constructs the assignment one variable at a time:

```text
BACKTRACK(assignment):
    if assignment contains every region:
        return assignment

    region <- choose an unassigned region

    for color in DOMAIN(region):
        if color is consistent with assignment:
            add region = color to assignment
            result <- BACKTRACK(assignment)

            if result is not FAILURE:
                return result

            remove region = color from assignment

    return FAILURE
```

### Algorithm 1 — Backtracking Search for a CSP

$$
\begin{array}{ll}
\textbf{function } \operatorname{BACKTRACK}(A) \\[2mm]

\quad \textbf{if } A \text{ is complete}
    & \textbf{return } A \\[1mm]

\quad X \leftarrow \operatorname{SELECT\_UNASSIGNED\_VARIABLE}(A) \\[1mm]

\quad \textbf{for each } v \in D(X) \\[1mm]

\qquad \textbf{if } \operatorname{CONSISTENT}(X,v,A) \\[1mm]

\qquad\quad A[X] \leftarrow v \\[1mm]

\qquad\quad R \leftarrow \operatorname{BACKTRACK}(A) \\[1mm]

\qquad\quad \textbf{if } R \neq \mathrm{FAILURE}
    & \textbf{return } R \\[1mm]

\qquad\quad \text{remove } X \text{ from } A \\[1mm]

\quad \textbf{return } \mathrm{FAILURE}
\end{array}
$$

where:

- $A$ is the current partial assignment,
- $X$ is an unassigned variable,
- $D(X)$ is the domain of possible values for $X$,
- $v \in D(X)$ is a candidate value,
- $\operatorname{CONSISTENT}(X,v,A)$ verifies that assigning $v$ to $X$ does not violate any constraint.

<div style="font-family: monospace; font-size: 15px; line-height: 1.6;
            background-color: #f6f8fa; padding: 18px 22px;
            border-radius: 8px; border-left: 4px solid #4a90e2;">

<b style="color:#2563eb;">BACKTRACK</b>(assignment):<br>
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">if</span> assignment contains every region:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">return</span> assignment<br>
<br>
&nbsp;&nbsp;&nbsp;&nbsp;region ← choose an unassigned region<br>
<br>
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">for</span> color <span style="color:#7c3aed;">in</span> DOMAIN(region):<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">if</span> color is consistent with assignment:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;add region = color to assignment<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;result ← <b style="color:#2563eb;">BACKTRACK</b>(assignment)<br>
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">if</span> result ≠ <span style="color:#dc2626;">FAILURE</span>:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">return</span> result<br>
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;remove region = color from assignment<br>
<br>
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:#7c3aed;">return</span> <span style="color:#dc2626;">FAILURE</span>

</div>

The algorithm prunes a branch as soon as a partial assignment violates a constraint. It backtracks when no available color can complete the current partial assignment.

### **3.2 The algorithm's inputs**

Mathematically, the algorithm receives:

- the variables in $\mathcal{X}$;
- the domain $D_i$ for each variable $X_i$;
- the neighbor relation that defines the constraints in $\mathcal{C}$.

The next section translates these mathematical objects into Python. The Python code starts only after the mathematical model and pseudocode are complete.

## **4. Write the problem data in Python**

We now translate the mathematical sets into Python objects, one step at a time. This cell defines data only; it does not search for a solution yet.

- `regions` stores the four variables.
- `colors` stores the domain of each variable.
- `neighboring_regions` stores the constraint relation.

In [13]:
regions = ["A", "B", "C", "D"]

colors = {
    "A": ["red", "blue", "green"],
    "B": ["red", "blue", "green"],
    "C": ["red", "blue", "green"],
    "D": ["red", "blue", "green"],
}

neighboring_regions = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}

## **5. Translate the mathematical model into Python**

Now we encode the three mathematical objects:

- `regions` represents the variables in $\mathcal{X}$.
- `colors` represents the domains in $\mathcal{D}$.
- `neighboring_regions` represents the relation used to check the constraints in $\mathcal{C}$.

The next cell implements the generic backtracking solver. It does not know that the variables are map regions; it only uses the three inputs.

```text
Assign A = red
      ↓
Try to solve the rest
      ↓
   ¿It works?
    /     \
  YES       NO
  ↓         ↓
return    delete A = red
            ↓
        try A = blue
```

In [ ]:
def solve_csp(variables, domains, neighbors):
    assignment = {}                              # Start with no assignments

    def backtrack():

        if len(assignment) == len(variables):    # All variables have a value
            return assignment

        for variable in variables:               # Choose an unassigned variable
            if variable not in assignment:
                break

        for value in domains[variable]:          # Try each possible value (domain, in order)
            valid = True                         # Flag to check if the assignment is valid

            for neighbor in neighbors[variable]: # Check the constraints
                if neighbor in assignment:       # If the neighbor is already assigned
                    if assignment[neighbor] == value: # If the neighbor has the same value, it's invalid
                        valid = False
                        break

            if valid == True:
                assignment[variable] = value     # Assign the value to the variable in the dictionary

                result = backtrack()             # Recursion

                if result is not None:
                    return result                # Solution found

                del assignment[variable]         # Undo assignment (backtrack)

        return None                              # No valid value


    result = backtrack()                         # Start the search
    return result

## **6. Run the solver on the map**

The data and the generic solver are now defined. We call the function by passing the three objects that correspond to the mathematical model:

The function returns a dictionary assigning one color to every region.

In [ ]:
coloring = solve_csp(regions, colors, neighboring_regions)
print("A valid coloring:", coloring)

A valid coloring: {'A': 'red', 'B': 'blue', 'C': 'green', 'D': 'red'}


<img src="csp_four_region_map_coloring.svg" alt="Solved map" width="400">

## **7. Check your understanding (optional)**

Use the map example, not an abstract definition, to answer these questions:

1. What are the four variables, and what is the domain of each variable?
2. Why is `{A: red, B: red}` rejected immediately?
3. Why is `{A: red, D: red}` allowed?
4. If A is `red` and B is `blue`, which colors remain possible for C?
5. What does the search return if no color is valid for an unassigned region?

## **8. Activity : course scheduling**

### Scenario

A university needs to schedule the following courses:

- Artificial Intelligence
- Calculus
- Algebra
- Programming
- Statistics
- Physics

The available time slots are:

- 08:00
- 10:00
- 12:00

Some courses cannot be scheduled at the same time because they have students in common.

### Constraints

The following pairs of courses cannot have the same time slot:

- Artificial Intelligence — Calculus
- Artificial Intelligence — Programming
- Artificial Intelligence — Statistics
- Calculus — Algebra
- Calculus — Physics
- Programming — Statistics
- Algebra — Physics

### Task

Using the backtracking algorithm studied in class:

1. Define the **variables**.
2. Define the **domains**.
3. Define the **constraints**.
4. Use the CSP solver from the class example to find a valid schedule.
5. Print the final schedule.

Do not modify the backtracking algorithm unless necessary.

### Submission

Submit:

- Your completed notebook.
- A maximum **2-minute individual video**, recorded during today's class where you explain your answers to the following questions.

### Questions

Answer briefly:

1. What are the variables in this CSP?
2. What is the domain?
3. Give an example of a constraint.
4. Is there more than one possible valid schedule? Explain briefly.
5. What happens if only two time slots are available?

### **References:**

For the formal definition of constraint satisfaction problems, variables, domains, constraints, backtracking search, and heuristics, read:

- Russell, S. and Norvig, P., *Artificial Intelligence: A Modern Approach*, 4th ed., Chapter 6, **Constraint Satisfaction Problems**.
- In particular, see Sections **6.1 (Defining Constraint Satisfaction Problems)** and **6.3 (Backtracking Search for CSPs)**.

The map-coloring problem in this notebook is the standard graph-coloring CSP discussed in that chapter; the presentation timetable is the same mathematical structure with different variable and domain names.